# 02 · Medallion pipeline: Turso → Bronze → Silver → Gold
| Step | What | Where |
|---|---|---|
| 1 | **Stage**: incremental extract from Turso, Parquet landing, append to Bronze as text | `bronze.*` |
| 2 | **Clean**: cast, standardise, DQ rules → quarantine, dedupe, guarded merge | `silver.*` |
| 3 | **Join**: sales ⟕ conformed dimensions, surrogate keys, Unknown member | in-flight |
| 4 | **Merge** into the Gold dimensions and fact | `gold.*` |
| 5 | **Delta** (optional): only Silver rows changed since the last Gold load | `ctl.watermark` |

Run it after each source batch (notebook 01). `LOAD_MODE=full` rebuilds Gold from all of Silver.

In [1]:
import os
os.chdir("/workspaces/pythonProject/medallion_v2/notebooks")

In [2]:
%run -i ./00_config.ipynb

cwd : /workspaces/pythonProject/medallion_v2/notebooks
ROOT: /workspaces/pythonProject/medallion_v2
LAKE: /workspaces/pythonProject/medallion_v2/lake
rules folder exists: True | file there: True
rule library: 22 checks for 8 entities


In [3]:
open_warehouse()
src = source_conn()
new_batch()
print(f"run batch {BATCH_ID} | gold load mode: {LOAD_MODE} | source: {'Turso' if TURSO_URL else 'local SQLite stand-in'}")

run batch 2 | gold load mode: incremental | source: Turso


## 1 · Stage (Bronze)

In [4]:
for e in ENTITIES: stage(e)

  customers  extracted     1 rows (since 2026-10-30 14:13:55),     0 new in bronze
  products   extracted     1 rows (since 2026-10-31 18:47:42),     0 new in bronze
  sales      extracted     1 rows (since 2026-10-31 20:05:30),     0 new in bronze


## 2 · Clean (Silver)

In [5]:
for e in ENTITIES: clean_to_silver(e)

  customers  nothing new in bronze
  products   nothing new in bronze
  sales      nothing new in bronze


## 3 · Join
Dimensions first (so keys exist), then join the sales delta to them.

In [6]:
for d in DIMS: load_dim(d)
print(f"  fact join   {build_fact_join():5d} rows ready (delta + any facts waiting on a late dimension)")

  dim_customer      0 inserted/updated
  dim_product       0 inserted/updated
  fact join      19 rows ready (delta + any facts waiting on a late dimension)


## 4 · Merge into Gold

In [7]:
print(f"  fact_sales  {merge_fact():5d} inserted/updated")

  fact_sales      0 inserted/updated


## 5 · Delta, watermarks & checks
Run the pipeline again with no new source data and every step reports "nothing new".

In [8]:
display(con.execute("SELECT name, value, updated_ts FROM ctl.watermark ORDER BY name").df())
display(con.execute("""
  SELECT 'bronze' AS layer, 'sales' AS tbl, count(*) AS n FROM bronze.sales UNION ALL
  SELECT 'silver', 'sales', count(*) FROM silver.sales UNION ALL
  SELECT 'silver', '_quarantine (all entities)', count(*) FROM silver._quarantine UNION ALL
  SELECT 'gold', 'fact_sales', count(*) FROM gold.fact_sales UNION ALL
  SELECT 'gold', 'fact rows with unknown dim key', count(*) FROM gold.fact_sales WHERE customer_sk = -1 OR product_sk = -1
""").df())

,name,value,updated_ts
0,extract.customers,2026-10-30 14:13:55,2026-11-18 23:31:12.973602
1,extract.products,2026-10-31 18:47:42,2026-11-18 23:31:12.995304
2,extract.sales,2026-10-31 20:05:30,2026-11-18 23:31:13.015919
3,gold.dim_customer,1,2026-11-18 23:31:13.104039
4,gold.dim_product,1,2026-11-18 23:31:13.114653
5,gold.fact_sales,1,2026-11-18 23:31:13.157791
6,silver.customers,1,2026-11-18 22:53:30.912295
7,silver.products,1,2026-11-18 22:53:30.954736
8,silver.sales,1,2026-11-18 22:53:31.015233


,layer,tbl,n
0,bronze,sales,304
1,silver,sales,300
2,silver,_quarantine (all entities),5
3,gold,fact_sales,300
4,gold,fact rows with unknown dim key,19


In [9]:
display(con.execute("SELECT entity, batch_id, _errors, payload FROM silver._quarantine ORDER BY batch_id, entity LIMIT 10").df())
display(con.execute("""
  SELECT f.order_date, c.name, c.segment, p.product_name, f.quantity, f.amount
  FROM gold.fact_sales f JOIN gold.dim_customer c USING (customer_sk) JOIN gold.dim_product p USING (product_sk)
  ORDER BY f.order_ts DESC LIMIT 10""").df())

,entity,batch_id,_errors,payload
0,products,1,"[""list_price_non_negative""]","{""product_id"":""19"",""product_name"":""Webcam"",""ca..."
1,sales,1,"[""order_ts_is_not_null""]","{""order_line_id"":""10091"",""order_id"":""5034"",""cu..."
2,sales,1,"[""quantity_positive""]","{""order_line_id"":""10008"",""order_id"":""5003"",""cu..."
3,sales,1,"[""quantity_positive""]","{""order_line_id"":""10042"",""order_id"":""5016"",""cu..."
4,sales,1,"[""unit_price_non_negative""]","{""order_line_id"":""10131"",""order_id"":""5048"",""cu..."


,order_date,name,segment,product_name,quantity,amount
0,2026-10-31,Jamal Silva,smb,Cable Pack,5,37.50
1,2026-10-31,Hector Brown,smb,Pro Kit,3,161.97
2,2026-10-31,Hector Brown,smb,Starter Kit,1,21.99
3,2026-10-31,Uma Garcia,retail,Cloud Seat,1,12.00
4,2026-10-31,Uma Nguyen,retail,Docking Station,3,348.30
5,2026-10-31,Bob Lee,enterprise,Analytics Add-on,3,161.70
6,2026-10-31,Hector Brown,smb,Backup Plan,4,60.00
7,2026-10-31,Uma Nguyen,retail,Gizmo,4,18.00
8,2026-10-31,Uma Garcia,retail,Widget,2,19.98
9,2026-10-31,Uma Nguyen,retail,Backup Plan,3,45.00


## 6 · Data-quality report (rules from `dq_rules.yml`)
Failed vs checked rows per rule, summed over all runs. `error` rules quarantine the row (see `_errors` above); `warn` rules only count.

In [10]:
display(con.execute("""SELECT entity, rule, criticality, function, sum(failed_rows) AS failed_rows, sum(checked_rows) AS checked_rows
                       FROM silver._dq_results GROUP BY ALL ORDER BY entity, criticality, rule""").df())

,entity,rule,criticality,function,failed_rows,checked_rows
0,customers,customer_id_is_not_null,error,is_not_null,0.0,60.0
1,customers,updated_at_is_not_null,error,is_not_null,0.0,60.0
2,customers,customer_email_format,warn,regex_match,0.0,60.0
3,customers,customer_segment_known,warn,is_in_list,0.0,60.0
4,products,list_price_is_not_null,error,is_not_null,0.0,20.0
5,products,list_price_non_negative,error,is_not_less_than,1.0,20.0
6,products,product_id_is_not_null,error,is_not_null,0.0,20.0
7,products,updated_at_is_not_null,error,is_not_null,0.0,20.0
8,sales,order_id_is_not_null,error,is_not_null,0.0,304.0
9,sales,order_line_id_is_not_null,error,is_not_null,0.0,304.0


### Optional · Prove the YAML runs unchanged on native DQX (Spark)
Runs the same `dq_rules.yml` with `DQEngineCore` on a local Spark session and compares the error / warning row counts with the DuckDB executor. Needs Java; if Spark cannot start the cell just prints why and is skipped.

In [11]:
for e in ENTITIES: native_dqx_check(e)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 23:31:15 WARN Utils: Your hostname, codespaces-f42fe3, resolves to a loopback address: 127.0.0.1; using 10.0.10.113 instead (on interface eth0)
26/11/18 23:31:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 23:31:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


  customers          DuckDB errors/warnings = 0/0 | native DQX = 0/0 -> MATCH
  products           DuckDB errors/warnings = 1/0 | native DQX = 1/0 -> MATCH
  sales              DuckDB errors/warnings = 4/0 | native DQX = 4/0 -> MATCH


## Scaling notes
* **Source volume:** extraction is keyset-style (`updated_at` watermark, ordered) in `CHUNK`-row Parquet parts, so memory stays flat. Use `TURSO_REMOTE=1` to avoid holding a local replica of a very large source.
* **Watermark caveat:** the extract assumes `updated_at` is set on every insert and update. Deletes are not seen; add an `is_deleted` flag or a tombstone table if you need them.
* **Warehouse size:** DuckDB comfortably handles hundreds of millions of rows on a Codespace; for more, point the same SQL at MotherDuck or write the layers as Parquet/Delta on object storage.
* **Orchestration:** the five steps are plain functions; call them from GitHub Actions on a schedule (`jupyter nbconvert --execute`) or from any scheduler.
* **Porting back to Databricks:** the metadata, step order and watermark logic carry over unchanged; the helpers become Auto Loader, Delta `MERGE` and Change Data Feed.